# 01 - EDA Sparkov : zone development uniquement

Les analyses portent uniquement sur les premiers 70 % chronologiques de
`fraudTrain.csv`. Les deux zones de calibration et le test final ne servent
pas aux decisions exploratoires. Aucun modele n'est entraine.

Les tableaux sont des agregats, jamais des apercus de transactions ou des
listes de personnes. Les fichiers sources restent inchanges. Les attributs
d'age et de genre sont analyses par groupes pour preparer l'audit d'equite.

Cette version est volontairement sans sorties enregistrees. Les observations
numeriques seront calculees lors d'une execution ulterieure, pas inventees ici.

## 1. Imports

On importe uniquement les bibliothèques utiles pour lire les données, calculer des statistiques simples et produire quelques graphiques.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "ml" / "src" / "data" / "load_data.py").is_file():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError(
            "Racine du projet introuvable : ouvrir le notebook depuis le projet."
        )
    PROJECT_ROOT = PROJECT_ROOT.parent

ML_SRC = PROJECT_ROOT / "ml" / "src"
if str(ML_SRC) not in sys.path:
    sys.path.insert(0, str(ML_SRC))

from data.load_data import load_sparkov_csv, split_chronologically
from preprocessing.pipeline import SparkovFeatureEngineer

EDA_COLUMNS = ["amt", "category", "gender", "state", "city_pop", "is_fraud"]
plt.style.use("seaborn-v0_8")

## 2. Chargement et zone development

Le chemin reste relatif a la racine du projet. Le chargeur lit uniquement
`fraudTrain.csv` ; le decoupage chronologique existant fournit la zone
`development` (70 %). Les zones de calibration ne sont pas analysees.
Le tableau de travail visible utilise une liste positive de colonnes utiles.

In [ ]:
TRAIN_PATH = PROJECT_ROOT / "data" / "raw" / "fraudTrain.csv"
print(TRAIN_PATH)
print(TRAIN_PATH.exists())
if not TRAIN_PATH.is_file():
    raise FileNotFoundError("Placer fraudTrain.csv dans data/raw avant de continuer.")

training_frame = load_sparkov_csv(TRAIN_PATH)
df_dev = split_chronologically(training_frame, sort_if_needed=True)["development"]
del training_frame
df = df_dev.loc[:, EDA_COLUMNS].copy()
print("Zone development, colonnes analysees :", df.shape)

## 3. Apercu agrege avec head()

On affiche les premieres categories agregees, pas des lignes de transactions.
Chaque ligne resume uniquement un effectif et un nombre de fraudes.

In [ ]:
df.groupby("category", observed=True)["is_fraud"].agg(
    transactions="size", fraudes="sum"
).head()

## 4. Dimensions avec shape

Ces dimensions concernent les lignes de development et les colonnes retenues
pour l'analyse, pas toutes les colonnes du fichier brut.

In [ ]:
df.shape

## 5. Structure avec info()

On examine seulement les types et effectifs des colonnes de travail autorisees.

In [ ]:
df.info()

## 6. Statistiques descriptives avec describe()

Les statistiques numeriques portent sur les montants, la population et la cible.
Aucune ligne individuelle n'est affichee.

In [ ]:
df.describe()

## 7. Types des colonnes

Cette étape aide à distinguer les colonnes numériques, textuelles et temporelles avant tout preprocessing.

In [ ]:
df.dtypes.sort_values()

## 8. Valeurs manquantes

Le comptage concerne les colonnes de travail. On ne remplace aucune valeur et
on ne supprime aucune ligne du jeu source.

In [ ]:
missing_values = df.isna().sum().sort_values(ascending=False)
missing_percent = (df.isna().mean() * 100).sort_values(ascending=False)

pd.DataFrame(
    {
        "valeurs_manquantes": missing_values,
        "pourcentage": missing_percent,
    }
).query("valeurs_manquantes > 0")

## 9. Doublons

On vérifie s'il existe des lignes totalement identiques. On ne supprime rien dans ce notebook.

In [ ]:
duplicate_count = int(df_dev.duplicated().sum())
duplicate_percent = duplicate_count / len(df_dev) * 100
print("Lignes completes dupliquees dans development :", duplicate_count)
print("Pourcentage :", duplicate_percent)

## 10. Distribution de `is_fraud`

`is_fraud` est la cible : `0` signifie transaction légitime, `1` signifie fraude.

In [ ]:
fraud_distribution = df["is_fraud"].value_counts().sort_index()
fraud_distribution

## 11. Nombre et pourcentage exact de fraudes

Ces valeurs sont calculees uniquement sur la zone development.

In [ ]:
fraud_count = int(df["is_fraud"].sum())
total_count = len(df)
fraud_percent = fraud_count / total_count * 100

print(f"Nombre total de transactions : {total_count}")
print(f"Nombre exact de fraudes : {fraud_count}")
print(f"Pourcentage exact de fraudes : {fraud_percent:.6f} %")

## 12. Statistiques sur `amt`

`amt` correspond au montant de la transaction. On observe sa dispersion, ses quartiles et ses valeurs extrêmes.

In [ ]:
df["amt"].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])

## 13. Distribution de `category`

La catégorie décrit le type de transaction. On regarde les catégories les plus fréquentes et leur taux de fraude.

In [ ]:
category_analysis = (
    df.groupby("category")
    .agg(
        transactions=("is_fraud", "size"),
        fraudes=("is_fraud", "sum"),
        taux_fraude=("is_fraud", "mean"),
    )
    .sort_values("transactions", ascending=False)
)

category_analysis

## 14. Analyse de `gender`

Le genre est conservé comme feature selon le document P2. Ici, on observe simplement la répartition et le taux de fraude par groupe.

In [ ]:
gender_analysis = (
    df.groupby("gender")
    .agg(
        transactions=("is_fraud", "size"),
        fraudes=("is_fraud", "sum"),
        taux_fraude=("is_fraud", "mean"),
    )
    .sort_values("transactions", ascending=False)
)

gender_analysis

## 15. Analyse de `state`

On observe les États avec le plus de transactions et les taux de fraude associés. Attention : un taux élevé sur peu de transactions peut être instable.

In [ ]:
state_analysis = (
    df.groupby("state")
    .agg(
        transactions=("is_fraud", "size"),
        fraudes=("is_fraud", "sum"),
        taux_fraude=("is_fraud", "mean"),
    )
    .sort_values("transactions", ascending=False)
)

state_analysis.head(15)

## 16. Analyse temporelle

La date sert uniquement a calculer les regroupements horaires et mensuels.
Seuls les agregats et les bornes globales de development sont affiches.

In [ ]:
transaction_dates = pd.to_datetime(
    df_dev["trans_date_trans_time"], format="%Y-%m-%d %H:%M:%S", errors="raise"
)
df_time = df.copy()
df_time["hour"] = transaction_dates.dt.hour
df_time["day_of_week"] = transaction_dates.dt.day_name()
df_time["month"] = transaction_dates.dt.to_period("M")

print("Borne temporelle minimale :", transaction_dates.min())
print("Borne temporelle maximale :", transaction_dates.max())
df_time.groupby("hour")["is_fraud"].agg(transactions="size", taux_fraude="mean")

## 17. Comparaison fraude / non-fraude pour le montant

On compare les montants selon la classe. Cela aide à voir si les fraudes ont des montants typiquement différents.

In [ ]:
df.groupby("is_fraud")["amt"].describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99])

## 18. Comparaison fraude / non-fraude selon l'heure

On compare la répartition horaire des transactions légitimes et frauduleuses.

In [ ]:
hour_class_analysis = df_time.groupby(["hour", "is_fraud"]).size().unstack(fill_value=0)
hour_class_analysis["taux_fraude"] = hour_class_analysis.get(
    1, 0
) / hour_class_analysis.sum(axis=1)
hour_class_analysis

## 19. Quelques visualisations simples

Ces graphiques donnent une lecture visuelle rapide : déséquilibre de la cible, montants, catégories, genre, États et heure de transaction.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

df["is_fraud"].value_counts().sort_index().plot(
    kind="bar", ax=axes[0, 0], title="Distribution de is_fraud"
)
axes[0, 0].set_xlabel("is_fraud")
axes[0, 0].set_ylabel("Nombre de transactions")

df["amt"].plot(kind="hist", bins=80, ax=axes[0, 1], title="Distribution de amt")
axes[0, 1].set_xlabel("Montant")

df.groupby("is_fraud")["amt"].median().plot(
    kind="bar", ax=axes[0, 2], title="Montant médian par classe"
)
axes[0, 2].set_xlabel("is_fraud")
axes[0, 2].set_ylabel("Montant médian")

df["category"].value_counts().head(10).plot(
    kind="bar", ax=axes[1, 0], title="Top 10 catégories"
)
axes[1, 0].set_xlabel("Catégorie")
axes[1, 0].set_ylabel("Nombre de transactions")

df.groupby("gender")["is_fraud"].mean().plot(
    kind="bar", ax=axes[1, 1], title="Taux de fraude par genre"
)
axes[1, 1].set_xlabel("Genre")
axes[1, 1].set_ylabel("Taux de fraude")

df_time.groupby("hour")["is_fraud"].mean().plot(
    kind="line", marker="o", ax=axes[1, 2], title="Taux de fraude par heure"
)
axes[1, 2].set_xlabel("Heure")
axes[1, 2].set_ylabel("Taux de fraude")

plt.tight_layout()
plt.show()

## Conclusions de l'EDA

La cellule suivante genere des observations a partir de development seulement.
Elle ne contient aucun resultat numerique fige. Les complements viennent ensuite.

In [ ]:
top_category = category_analysis["transactions"].idxmax()
top_category_count = int(category_analysis.loc[top_category, "transactions"])
highest_fraud_category = category_analysis["taux_fraude"].idxmax()
highest_fraud_category_rate = (
    category_analysis.loc[highest_fraud_category, "taux_fraude"] * 100
)

median_amt_legit = df.loc[df["is_fraud"] == 0, "amt"].median()
median_amt_fraud = df.loc[df["is_fraud"] == 1, "amt"].median()

peak_fraud_hour = df_time.groupby("hour")["is_fraud"].mean().idxmax()
peak_fraud_hour_rate = df_time.groupby("hour")["is_fraud"].mean().max() * 100

missing_total = int(df.isna().sum().sum())

conclusions = f"""
- La zone `development` contient **{total_count:,} transactions** et **{df.shape[1]} colonnes analysees**.
- La repartition de la cible est : **{fraud_count:,} fraudes**, soit **{fraud_percent:.6f} %** des transactions.
- Les colonnes analysees contiennent **{missing_total:,} valeurs manquantes** au total.
- Le nombre de lignes totalement dupliquées est **{duplicate_count:,}**, soit **{duplicate_percent:.6f} %** du dataset.
- La catégorie la plus fréquente est **`{top_category}`** avec **{top_category_count:,} transactions**.
- La catégorie avec le taux de fraude le plus élevé est **`{highest_fraud_category}`** avec **{highest_fraud_category_rate:.6f} %**.
- Le montant médian des transactions légitimes est **{median_amt_legit:.2f}**, contre **{median_amt_fraud:.2f}** pour les transactions frauduleuses.
- L'heure avec le taux de fraude le plus élevé est **{peak_fraud_hour}h**, avec **{peak_fraud_hour_rate:.6f} %**.
- Ces observations servent uniquement à comprendre les données ; elles ne constituent pas encore du preprocessing ni du Machine Learning.
"""

display(Markdown(conclusions))

## 20. EDA complementaire P2

Les complements ci-dessous utilisent uniquement development. Le transformeur
deterministe existant fournit les variables derivees sans fit ni mise a
l'echelle. Il n'y a ni modele, ni decision fondee sur les zones reservees.

In [ ]:
def wilson_interval(successes, n, z=1.96):
    """Retourne un IC de Wilson a 95 %, ou NaN pour un groupe vide."""
    successes = np.asarray(successes, dtype="float64")
    n = np.asarray(n, dtype="float64")
    safe_n = np.where(n > 0, n, np.nan)
    proportion = successes / safe_n
    denominator = 1 + z**2 / safe_n
    center = (proportion + z**2 / (2 * safe_n)) / denominator
    half = (
        z
        * np.sqrt((proportion * (1 - proportion) + z**2 / (4 * safe_n)) / safe_n)
        / denominator
    )
    return center - half, center + half


feature_engineer = SparkovFeatureEngineer()
derived_features = feature_engineer.transform(df_dev)
eda_dev = df.copy()
for column in ["age", "distance_km", "amount_log", "city_pop_log", "hour", "is_night"]:
    eda_dev[column] = derived_features[column]
del derived_features
eda_dev["month"] = df_time["month"]
eda_dev["age_group"] = pd.cut(
    eda_dev["age"],
    bins=[0, 25, 35, 45, 55, 65, 120],
    labels=["0-25", "26-35", "36-45", "46-55", "56-65", "66-120"],
    include_lowest=True,
)
print("Effectif development :", len(eda_dev))
print("Fraudes development :", int(eda_dev["is_fraud"].sum()))

### 20.1 `state` : cardinalité, fraudes et intervalle de Wilson

On regarde la cardinalité des États et le taux de fraude avec intervalle de Wilson à 95 %. Cela évite de surinterpréter un taux élevé sur un petit effectif.

In [ ]:
state_cardinality = eda_dev["state"].nunique(dropna=False)
state_wilson = (
    eda_dev.groupby("state")
    .agg(
        transactions=("is_fraud", "size"),
        fraudes=("is_fraud", "sum"),
        taux_fraude=("is_fraud", "mean"),
    )
    .sort_values(["fraudes", "transactions"], ascending=False)
)
state_wilson["wilson_low"], state_wilson["wilson_high"] = wilson_interval(
    state_wilson["fraudes"], state_wilson["transactions"]
)
print("Cardinalité state :", state_cardinality)
state_wilson.head(15)

### 20.2 `merchant` et `job` : cardinalité et taux par modalité

Ces colonnes sont candidates à l'exclusion dans le document P2 à cause de leur forte cardinalité et du risque de sur-apprentissage. On chiffre cette cardinalité sur `development`.

In [ ]:
merchant_cardinality = df_dev["merchant"].nunique(dropna=False)
job_cardinality = df_dev["job"].nunique(dropna=False)


def ranked_modality_rates(column):
    """Agrege les modalites sans retourner leurs libelles dans les tableaux."""
    rates = df_dev.groupby(column, observed=True)["is_fraud"].agg(
        transactions="size", fraudes="sum", taux_fraude="mean"
    )
    return (
        rates.sort_values(["fraudes", "transactions"], ascending=False)
        .reset_index(drop=True)
        .rename_axis("rang_modalite")
    )


print("Cardinalite merchant :", merchant_cardinality)
print("Cardinalite job :", job_cardinality)
display(ranked_modality_rates("merchant").head(10))
display(ranked_modality_rates("job").head(10))

### 20.3 Age : distribution et taux de fraude par tranche

L'age en annees revolues vient du transformeur de production. Seules sa
distribution agregee et les statistiques par tranche sont presentees.

In [ ]:
age_distribution = eda_dev["age"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
)
age_group_rates = eda_dev.groupby("age_group", observed=False).agg(
    transactions=("is_fraud", "size"),
    fraudes=("is_fraud", "sum"),
    taux_fraude=("is_fraud", "mean"),
)
display(age_distribution)
age_group_rates

### 20.4 `distance_km`, `amount_log`, `city_pop_log` par classe

On compare les distributions de features dérivées prévues dans P2 entre fraude et non-fraude.

In [ ]:
derived_by_class = eda_dev.groupby("is_fraud")[
    ["distance_km", "amount_log", "city_pop_log"]
].describe(percentiles=[0.25, 0.5, 0.75, 0.95])
derived_by_class

### 20.5 Dérive temporelle : taux de fraude par mois

On vérifie si le taux de fraude varie au cours du temps dans la zone `development`.

In [ ]:
monthly_fraud = eda_dev.groupby("month").agg(
    transactions=("is_fraud", "size"),
    fraudes=("is_fraud", "sum"),
    taux_fraude=("is_fraud", "mean"),
)
monthly_fraud

### 20.6 Heure et comparaison des plages nocturnes

On compare 22-3 h, 22-5 h et 0-5 h avec effectifs et IC de Wilson.
Ce diagnostic ne remplace pas automatiquement la definition du pipeline,
qui reste 22-3 h. Aucun choix n'est fait sur une zone reservee.

In [ ]:
hour_rates = eda_dev.groupby("hour").agg(
    transactions=("is_fraud", "size"),
    fraudes=("is_fraud", "sum"),
    taux_fraude=("is_fraud", "mean"),
)
hour_rates["wilson_low"], hour_rates["wilson_high"] = wilson_interval(
    hour_rates["fraudes"], hour_rates["transactions"]
)

night_rows = []
for label, hours in [
    ("22-3", [22, 23, 0, 1, 2, 3]),
    ("22-5", [22, 23, 0, 1, 2, 3, 4, 5]),
    ("0-5", [0, 1, 2, 3, 4, 5]),
]:
    mask = eda_dev["hour"].isin(hours)
    fraudes = int(eda_dev.loc[mask, "is_fraud"].sum())
    transactions = int(mask.sum())
    low, high = wilson_interval([fraudes], [transactions])
    night_rows.append(
        {
            "definition": label,
            "transactions": transactions,
            "fraudes": fraudes,
            "taux_fraude": fraudes / transactions if transactions else np.nan,
            "wilson_low": float(low[0]),
            "wilson_high": float(high[0]),
        }
    )
night_definition_compare = pd.DataFrame(night_rows).sort_values(
    "taux_fraude", ascending=False
)

display(hour_rates)
night_definition_compare

### 20.7 Visualisations complémentaires

On ajoute deux graphiques simples : histogramme de `amt` en échelle log et boxplot du montant par classe.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

eda_dev["amount_log"].plot(
    kind="hist", bins=80, ax=axes[0], title="Distribution de log(1 + amt)"
)
axes[0].set_xlabel("log(1 + montant)")

eda_dev.boxplot(column="amt", by="is_fraud", ax=axes[1])
axes[1].set_title("Montant par classe")
axes[1].set_xlabel("is_fraud")
axes[1].set_ylabel("amt")
plt.suptitle("")
plt.tight_layout()
plt.show()

## Decisions de pretraitement

La synthese cite uniquement des valeurs calculees dans development. Elle
rappelle les choix existants du pipeline, sans les redecider automatiquement
en fonction d'un taux maximal. Toute nouvelle option doit etre validee.

In [ ]:
amount_corr = float(eda_dev[["amt", "amount_log"]].corr(method="spearman").iloc[0, 1])
monthly_min = float(monthly_fraud["taux_fraude"].min())
monthly_max = float(monthly_fraud["taux_fraude"].max())
age_rates = age_group_rates["taux_fraude"].dropna()
age_pattern = (
    "non monotone"
    if (
        len(age_rates) > 1
        and not age_rates.is_monotonic_increasing
        and not age_rates.is_monotonic_decreasing
    )
    else "monotone ou effectifs insuffisants pour conclure"
)
distance_medians = eda_dev.groupby("is_fraud")["distance_km"].median()
night_mask = eda_dev["is_night"].eq(1)
night_count = int(night_mask.sum())
night_frauds = int(eda_dev.loc[night_mask, "is_fraud"].sum())
night_rate = night_frauds / night_count if night_count else np.nan
night_low, night_high = wilson_interval([night_frauds], [night_count])

preprocessing_decisions = f"""
- `category` : {eda_dev["category"].nunique()} modalites ; encodage du pipeline conserve.
- `state` : {state_cardinality} modalites ; les intervalles affiches contextualisent les taux.
- Cardinalites exclues du modele actuel : merchant={merchant_cardinality}, job={job_cardinality}.
- Nuit du pipeline {feature_engineer.night_hours} : {night_count} transactions, {night_frauds} fraudes,
  taux {night_rate:.6f}, IC Wilson [{night_low[0]:.6f}, {night_high[0]:.6f}].
- Distance mediane legitime : {distance_medians.get(0, np.nan):.2f} km ;
  fraude : {distance_medians.get(1, np.nan):.2f} km. La distance derivee reste conservee.
- Profil des taux par tranche d'age : {age_pattern}. Une forme en U ne se deduit pas du seul minimum.
- Taux mensuel entre {monthly_min:.6f} et {monthly_max:.6f} sur development.
- Redondance montant / log : correlation de Spearman {amount_corr:.6f} ; le pipeline garde amount_log.
- city_pop_log, age et gender restent conformes au pipeline actuel ; audit d'equite ulterieur.
"""
display(Markdown(preprocessing_decisions))